# 3. Modern topic inventories, publication patterns, and PR3 evidence
**Question:** How does the selected BERTopic model describe coverage across outlets?

This notebook exports modern-only results for the grid and the selected run. The primary
analysis describes this fixed corpus; it is not a held-out prediction test. Interpret small
outlets, unequal date coverage, boilerplate/transcript clusters, and keyword-filtering bias cautiously.

The separate [pipeline comparison notebook](../../comparison/pipeline_comparison.ipynb)
handles joint evaluation.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
ROOT = REPO/'final-project/code/modern-methods'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from shared.corpus import corpus_hash, digest
from shared import evaluation as ev
from src import topic_helpers as h, artifacts as a
OUT = ROOT/'Results/modern'
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
df, embeddings = h.load_inputs(ROOT)
reservation = json.loads((OUT/'review_sample.json').read_text())
assert reservation['corpus_hash'] == corpus_hash(df)
experiments = pd.read_csv(OUT/'experiments.csv')
selection = a.selection_record(OUT, df, experiments.run_id.tolist())
notebook = json.loads((ROOT/'notebooks/02_modern_topic_modeling.ipynb').read_text())
fit_source = ''.join(''.join(c['source']) for c in notebook['cells'] if 'model-fit' in c.get('metadata',{}).get('tags',[]))
current_implementation = digest([fit_source] + [(ROOT/p).read_text() for p in ['src/artifacts.py','src/topic_helpers.py','../shared/evaluation.py']])
# Verify current data and artifacts before exporting any run's evidence.
for run_id in experiments.run_id:
    folder = OUT/'runs'/run_id
    model_check, metrics_check = a.load_run(folder)
    identity = json.loads((folder/'manifest.json').read_text())['identity']
    assert identity['corpus_hash'] == corpus_hash(df) and identity['embedding_hash'] == a.array_hash(embeddings)
    assert identity['reservation_hash'] == digest(reservation)
    if identity['implementation_hash'] != current_implementation or identity['versions'] != a.environment()['versions']:
        raise ValueError('Experiment code or environment changed; rerun notebook 02 after preserving old runs')
    assignments = pd.read_csv(folder/'assignments.csv',dtype={'article_id':str})
    assert assignments.topic_id.tolist() == model_check.topics_
    inventory = a.topic_inventory(folder, folder, len(df))
    ev.publication_exports(df, assignments, folder/'publication',dict(zip(inventory.topic_id,inventory.label)))
print('Per-run topic inventories and publication outputs are complete.')

/Users/marykate/Desktop/F26/DataMiningProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Per-run topic inventories and publication outputs are complete.


## Selected model and publication comparisons
CSV matrices contain all 25 publications and every topic. The main heatmap includes outlets
with at least 50 articles and the 15 largest topics plus −1. Its cells are fractions of **all**
articles in each outlet, including unassigned articles. The displayed subset need not sum to one.
These are descriptive within-corpus differences, not estimates of all coverage by an outlet.

In [3]:
modern = terms = inventory = labels = None
labels_ready = False
SELECTED = None
if selection is None:
    print('Selected-model interpretation pending: fill selection.json and rerun notebook 02.')
    display(experiments[['run_id','topic_count','coherence','outlier_fraction']])
else:
    run_id = selection['run_id']
    folder = OUT/'runs'/run_id
    SELECTED = OUT/'selected'/run_id
    SELECTED.mkdir(parents=True, exist_ok=True)
    manifest = json.loads((folder/'manifest.json').read_text())
    labels = a.labels_table(folder, SELECTED, {'run_id':run_id,'corpus_hash':corpus_hash(df),'run_identity_hash':digest(manifest['identity'])})
    labels_ready = bool(len(labels) and labels.confirmed.astype(str).str.lower().eq('true').all())
    modern = pd.read_csv(folder/'assignments.csv',dtype={'article_id':str})
    terms = pd.read_csv(folder/'terms.csv')
    inventory = a.topic_inventory(folder, SELECTED, len(df), labels)
    display(inventory[['topic_id','label','label_status','count','prevalence','coherence']])
    counts, proportions = ev.publication_exports(df, modern, SELECTED,dict(zip(inventory.topic_id,inventory.label)))
    from IPython.display import Image
    display(Image(filename=str(SELECTED/'publication_proportions.png')))
    if (SELECTED/'stability.csv').exists():
        display(pd.read_csv(SELECTED/'stability.csv'))
    else:
        print('Seed stability pending: rerun notebook 02 after selection.')
    if not labels_ready:
        print('Topic naming pending team confirmation; numeric outputs use provisional keyword labels.')

Selected-model interpretation pending: fill selection.json and rerun notebook 02.


,run_id,topic_count,coherence,outlier_fraction
0,n15_c25_s42,76,0.692091,0.202400
1,n15_c50_s42,40,0.662209,0.199264
2,n15_c100_s42,21,0.640805,0.146072
3,n30_c25_s42,70,0.704696,0.180442
4,n30_c50_s42,35,0.645542,0.190807
5,n30_c100_s42,24,0.629542,0.222040


## Modern evidence for PR3
Use actual output tables and captions to update the manuscript manually. This notebook does
not edit `pr3-submission.tex`. Separate quantitative results, team interpretations, and unfinished work.

- **Grid table:** Six configurations on 7,332 deduplicated articles; coherence uses the shared reference; silhouette excludes unassigned articles.
- **Topic inventory:** Sizes, top-ten terms, coherence, and development representatives. Only confirmed labels support named thematic claims.
- **Publication heatmap:** Within-outlet shares of all articles; n≥50 outlets, 15 largest topics plus outliers; full matrices are exported separately.
- **Stability:** ARI conditional on assignment in both runs; matched publication changes, unmatched topics, and split/merge flags (two overlaps each covering at least 10% of the source/target topic).

Which themes are supported by the development articles? Which outlets differ in topic share,
and could date coverage explain it? Which clusters reflect transcript style or marginal relevance?
Do matched seed topics support the same publication patterns?

In [4]:
evidence = pd.DataFrame([
    ('Data / methodology','Corpus, exclusions, embedding settings','complete','corpus_by_publication.csv; review_sample.json; cache manifest'),
    ('Experimental design / results','Six-configuration study','complete','experiments.csv'),
    ('Results / discussion','Team model choice','pending' if selection is None else 'complete','selection.json'),
    ('Results / discussion','Topic labels and article support','complete' if labels_ready else 'pending','selected/<run>/topic_labels.csv; topic_inventory.csv'),
    ('Results','Publication patterns','all grid runs complete; selected interpretation pending' if selection is None else 'selected run complete','runs/<run>/publication; selected/<run>/publication_proportions.csv'),
    ('Robustness','Selected configuration seeds 7 and 21','complete' if SELECTED is not None and (SELECTED/'stability.csv').exists() else 'pending','selected/<run>/stability.csv; seed_*/publication_changes.csv'),
    ('Limitations','Corpus bias, coverage, pooling, outliers','documented; team interpretation required','Notebook discussion prompts'),
    ('Response to feedback','PR2 feedback','not available in repository','Team must supply actual feedback')
],columns=['manuscript_section','evidence','status','artifact'])
evidence.to_csv(OUT/'pr3_evidence_checklist.csv',index=False)
display(evidence)
completion = pd.DataFrame([
    ('Confirm modern selection and topic labels','Needed for thematic findings','Mary Kate + team','selection rationale and confirmed labels','Team to confirm'),
    ('Run chosen configuration with seeds 7 and 21','Assess assignment and publication-pattern stability','Mary Kate','ARI, outlier agreement, topic matches and publication changes','Team to confirm'),
    ('Modern held-out evaluation with rare-stratum policy','Assess transfer beyond fitted articles','Mary Kate','training-only fit and held-out metrics','Team to confirm'),
    ('Modern shared-period publication sensitivity','Assess unequal date coverage','Mary Kate + Lauren','date-restricted publication tables','Team to confirm'),
    ('Interpret findings and actual feedback','Support manuscript revision','Team','article-supported interpretations','Team to confirm')
],columns=['remaining_task','reason','responsible_member','expected_output','completion_date'])
completion.to_csv(OUT/'completion_plan.csv',index=False)
display(completion)

,manuscript_section,evidence,status,artifact
0,Data / methodology,"Corpus, exclusions, embedding settings",complete,corpus_by_publication.csv; review_sample.json;...
1,Experimental design / results,Six-configuration study,complete,experiments.csv
2,Results / discussion,Team model choice,pending,selection.json
3,Results / discussion,Topic labels and article support,pending,selected/<run>/topic_labels.csv; topic_invento...
4,Results,Publication patterns,all grid runs complete; selected interpretatio...,runs/<run>/publication; selected/<run>/publica...
5,Robustness,Selected configuration seeds 7 and 21,pending,selected/<run>/stability.csv; seed_*/publicati...
6,Limitations,"Corpus bias, coverage, pooling, outliers",documented; team interpretation required,Notebook discussion prompts
7,Response to feedback,PR2 feedback,not available in repository,Team must supply actual feedback


,remaining_task,reason,responsible_member,expected_output,completion_date
0,Confirm modern selection and topic labels,Needed for thematic findings,Mary Kate + team,selection rationale and confirmed labels,Team to confirm
1,Run chosen configuration with seeds 7 and 21,Assess assignment and publication-pattern stab...,Mary Kate,"ARI, outlier agreement, topic matches and publ...",Team to confirm
2,Modern held-out evaluation with rare-stratum p...,Assess transfer beyond fitted articles,Mary Kate,training-only fit and held-out metrics,Team to confirm
3,Modern shared-period publication sensitivity,Assess unequal date coverage,Mary Kate + Lauren,date-restricted publication tables,Team to confirm
4,Interpret findings and actual feedback,Support manuscript revision,Team,article-supported interpretations,Team to confirm


## Deferred analyses and responsible interpretation
The held-out experiment and shared-period sensitivity remain final-report work. Some publication/year
strata contain one article, so strict stratification cannot put every stratum in both partitions.
Document a rare-stratum policy before splitting. A genuine held-out study requires training-only
vocabulary learning, fitting, and selection; these exploratory full-corpus results are not test performance.

No sentiment, public-opinion, or causal claims follow from these topic assignments. PR2 feedback is
not present locally, so no claim is made that it has been addressed. Record substantive AI assistance
with coding and planning separately from human research judgments.